## Tylko testy do większych zbiorów danych

In [1]:
import os
import pandas as pd
from torchvision.io import read_image
import re
import wfdb
import wfdb.processing
import scipy
from torch.utils.data import Dataset
import numpy as np
import json
import torch.nn as nn
import torch
from tqdm import tqdm
import torch.nn.functional as F

In [2]:
def extract_segment_with_padding(z, k, N):
    # Rozmiar segmentu to 2N + 1
    start_idx = k - N
    end_idx = k + N + 1  # Indeks końcowy +1, ponieważ Python używa wykluczającego indeksu
    
    # Upewnij się, że start_idx i end_idx mieszczą się w granicach tablicy
    if start_idx < 0:
        # Jeśli start_idx jest poza zakresem, dopełnij na początku
        padding_left = np.median(z[:end_idx])  # Wypełniamy medianą
        segment = np.concatenate([np.full(-start_idx, padding_left), z[:end_idx]])
    elif end_idx > len(z):
        # Jeśli end_idx jest poza zakresem, dopełnij na końcu
        padding_right = np.median(z[start_idx:])  # Wypełniamy medianą
        segment = np.concatenate([z[start_idx:], np.full(end_idx - len(z), padding_right)])
    else:
        # Normalny przypadek, kiedy zakres mieści się w tablicy
        segment = z[start_idx:end_idx]
    
    return segment

def find_nearest_qrs_index(annotation_sample, qrs_inds):
    # Find the index in qrs_inds that is closest to annotation_sample
    distances = np.abs(qrs_inds - annotation_sample)
    nearest_idx = np.argmin(distances)  # Get the index of the minimum distance
    return qrs_inds[nearest_idx]

class MIT_BIH_Arythmia(Dataset):
    def __init__(self,N, M, dataset_dir = 'Datasets/files/', fs = 10, filename = "MIT-BIH_Arrythmia.json"):
        """
        n - number of samples of orginal signal resampled to fs, interval [-n,n]
        m - qrs times, interval [-m,m]
        """
        self.N = N
        self.ecg_list = []
        exclusion_lst = ["00735", "03665", "04043", "04936", "05091", "06453", "08378", "08405", "08434", "08455"]
        for file in os.listdir(dataset_dir):
            name = re.match(r'^(.*\d\d+)\.atr$', file)
            if name:
                if name.group(1) in exclusion_lst:
                    continue
            if name:
                record = wfdb.rdsamp(f"{dataset_dir}{name.group(1)}") 
                annotation = wfdb.rdann(f"{dataset_dir}{name.group(1)}", 'atr')
                signal = record[0][:,0]
                ####
                fs = record[1]["fs"]
                cutOff = 20
                b, a = scipy.signal.butter(5, cutOff, fs=fs, btype='low', analog=False)
                signal = scipy.signal.lfilter(b,a,signal)
                ####


                fs_original = record[1]["fs"]
                num_samples_target = int(signal.shape[0] * fs / fs_original)
                resampled_signal = scipy.signal.resample(signal, num_samples_target)
                annotation_times_resampled = (annotation.sample * fs) / fs_original
                resampled_annotation = wfdb.Annotation('atr',annotation.symbol,annotation_times_resampled.astype(int),aux_note=annotation.aux_note)
                self.ecg_list.append({"name": name.group(1),"rec" : resampled_signal, "ann" : resampled_annotation})
        self.samples_list = []
        self.label_list = []
        self.qrs_samples = []
        self.idx = []
        self.label = []
        self.number = []
        no_of_afib = 0
        no_of_normal = 0
        for n,dic in enumerate(self.ecg_list):
            print(dic["name"])
            # xqrs = wfdb.processing.XQRS(sig=dic["rec"], fs=fs)
            # xqrs.detect()
            # qrs_inds = xqrs.qrs_inds
            if(len(dic["ann"].sample)==1):
                idx_next = len(dic["rec"])
            else:
                idx_next = dic["ann"].sample[1]
            label_t = dic["ann"].aux_note[0]
            temp_aux = 0
            for idx in range(dic["ann"].sample[0],len(dic["rec"]),20):
                if(idx>=idx_next):
                    temp_aux += 1
                    if(temp_aux!=len(dic["ann"].sample)-1):
                        idx_next = dic["ann"].sample[temp_aux+1]
                    else:
                        idx_next = len(dic["rec"])
                    label_t = dic["ann"].aux_note[temp_aux]
                
                self.label.append(1 if label_t == '(AFIB' else 0)
                if (label_t == '(AFIB'):
                    no_of_afib+=1
                else:
                    no_of_normal+=1
                
                self.idx.append(idx)
                self.number.append(n)

        print(no_of_afib, no_of_normal)
        #     for n,i in enumerate(dic["ann"].sample):
        #         self.label_list.append(1 if dic["ann"].aux_note[n] == '(AFIB' else 0)
        #         self.samples_list.append(list(extract_segment_with_padding(dic["rec"], dic["ann"].sample[n],N)))
        #         # nearest_qrs_idx = find_nearest_qrs_index(dic["ann"].sample[n], qrs_inds)
        #         # self.qrs_samples.append(list(extract_segment_with_padding(qrs_inds,nearest_qrs_idx,M)))
        # data = {
        #     'samples_list': self.samples_list,  # This would work if the segments are simple numeric lists
        #     'label_list': self.label_list,
        #     'qrs_samples': self.qrs_samples
        # }
        # with open(filename, 'w') as f:
        #     json.dump(data, f)
                
    def __len__(self):
        return len(self.idx)

    def __getitem__(self, idx):
        sample = list(extract_segment_with_padding(self.ecg_list[self.number[idx]]["rec"], self.idx[idx],self.N))
        data = torch.Tensor(sample).unsqueeze(0)
        return data, self.label[idx]

In [4]:
ds = MIT_BIH_Arythmia(100,5,fs=20)

04015
04048
04126
04746
04908
05121
05261
06426
06995
07162
07859
07879
07910
08215
08219
600271 780588


In [5]:


class SimpleConv(nn.Module):
    def __init__(self, input = 201, input_ch = 1, num_classes = 2):
        super(SimpleConv, self).__init__()
        self.model = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding='same'),     
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding='same'),     
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear(128*(input//2), 256),
            nn.Sigmoid(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        self.model.to('cuda:0')

    def forward(self, x):

        return self.model(x)
    
    def train_model(self, train_loader, valid_loader, num_epochs = 5, learning_rate=0.001, save_best = False, save_thr = 0.94):
        best_accuracy = 0.0
        total_step = len(train_loader)
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.RMSprop(self.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)  

        for epoch in range(num_epochs):
            # self.train()
            correct = 0
            total = 0
            for i, (images, labels) in enumerate(tqdm(train_loader)):
                # Move tensors to the configured device
                images = images.float().to("cuda")
                labels = labels.type(torch.LongTensor)
                labels = labels.to("cuda")


                optimizer.zero_grad()

                # Forward pass
                outputs = self.forward(images)
                loss = criterion(outputs, labels)
                # Backward and optimize
                loss.backward()
                
                optimizer.step()

                # accuracy
                _, predicted = torch.max(outputs.data, 1)
                correct += (torch.eq(predicted, labels)).sum().item()
                total += labels.size(0)

                del images, labels, outputs

            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
                            .format(epoch+1, num_epochs, i+1, total_step, loss.item(), (float(correct))/total))


            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Validation
            with torch.no_grad():
                correct = 0
                total = 0
                for images, labels in valid_loader:
                    images = images.float().to("cuda")
                    labels = labels.to("cuda")
                    outputs = self.forward(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (torch.eq(predicted, labels)).sum().item()
                    del images, labels, outputs
                if(((100 * correct / total) > best_accuracy) and save_best and ((100 * correct / total) > save_thr)):
                    torch.save(self.state_dict(), "best_simple_afdb.pt")

                print('Accuracy of the network: {} %'.format( 100 * correct / total))

In [6]:
model = SimpleConv()

In [11]:
from torch.utils.data import DataLoader, random_split
train_set, val_set = random_split(ds, [0.8, 0.2])
train = DataLoader(train_set, batch_size=32, shuffle=True)
val = DataLoader(val_set, batch_size=32, shuffle=True)

In [7]:
model.train_model(train,val,num_epochs=90, save_best=True)

100%|██████████| 34522/34522 [01:57<00:00, 293.53it/s]


Epoch [1/90], Step [34522/34522], Loss: 0.6636, Accuracy: 0.6063
Accuracy of the network: 61.3677757621184 %


100%|██████████| 34522/34522 [01:36<00:00, 356.80it/s]


Epoch [2/90], Step [34522/34522], Loss: 0.4965, Accuracy: 0.6045
Accuracy of the network: 56.53019324983434 %


100%|██████████| 34522/34522 [01:36<00:00, 356.83it/s]


...

Epoch [7/90], Step [34522/34522], Loss: 0.6393, Accuracy: 0.6225
Accuracy of the network: 58.64084208696786 %


100%|██████████| 34522/34522 [01:36<00:00, 357.33it/s]


Epoch [8/90], Step [34522/34522], Loss: 0.5539, Accuracy: 0.6243
Accuracy of the network: 57.510020965271515 %


 29%|██▉       | 10074/34522 [00:28<01:08, 356.84it/s]


KeyboardInterrupt: 

In [8]:
model.train_model(train,val,num_epochs=90, save_best=True, learning_rate=0.0001)

100%|██████████| 34522/34522 [01:41<00:00, 340.95it/s]


Epoch [1/90], Step [34522/34522], Loss: 0.5892, Accuracy: 0.6753
Accuracy of the network: 68.80700725275283 %


100%|██████████| 34522/34522 [01:36<00:00, 356.13it/s]


Epoch [2/90], Step [34522/34522], Loss: 0.7116, Accuracy: 0.6780
Accuracy of the network: 67.05446987554812 %


100%|██████████| 34522/34522 [01:36<00:00, 357.60it/s]


...

Epoch [4/90], Step [34522/34522], Loss: 0.5775, Accuracy: 0.6800
Accuracy of the network: 67.81740298583124 %


100%|██████████| 34522/34522 [01:35<00:00, 359.70it/s]


Epoch [5/90], Step [34522/34522], Loss: 0.7198, Accuracy: 0.6799
Accuracy of the network: 68.4148589098783 %


 47%|████▋     | 16391/34522 [00:45<00:50, 359.77it/s]


KeyboardInterrupt: 

In [10]:
model.train_model(train,val,num_epochs=90, save_best=True, learning_rate=0.00001, save_thr=0.68)

100%|██████████| 34522/34522 [02:11<00:00, 262.66it/s]


Epoch [1/90], Step [34522/34522], Loss: 0.6550, Accuracy: 0.7043
Accuracy of the network: 70.62363535635531 %


  9%|▉         | 3110/34522 [00:12<02:03, 254.17it/s]


KeyboardInterrupt: 

## Model a La resnet

In [6]:
class ResNetBlock(nn.Module):
    def __init__(self,in_channels, out_channels):
        """
        output same as input
        """
        super(ResNetBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
                        nn.BatchNorm1d(out_channels),
                        nn.ReLU(inplace=False))  # Changed inplace to False
        self.conv2 = nn.Sequential(
                        nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
                        nn.BatchNorm1d(out_channels),
                        nn.ReLU(inplace=False))
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        if(in_channels != out_channels):
            self.residual = nn.Sequential(
                nn.Conv1d(self.in_channels, out_channels, kernel_size=1, stride=1),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self,x):
        out = self.conv1(x)
        out = self.conv2(out)
        if self.in_channels != self.out_channels:
            residual = self.residual(x)
        else:
            residual = x
        return F.relu(out + residual, inplace=False)


class ResNetLike(nn.Module):
    def __init__(self, input = 201, input_ch = 1, num_classes = 2):
        super(ResNetLike, self).__init__()
        self.model = nn.Sequential(
            nn.Conv1d(input_ch, 64, kernel_size=7, padding='same'),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            ResNetBlock(64,64),
            ResNetBlock(64,64),
            ResNetBlock(64,64),
            ResNetBlock(64,128),    # out 1 x 128 x n
            nn.MaxPool1d(2),        # out 1 x 128 x n//2
            ResNetBlock(128,128),
            ResNetBlock(128,128),
            ResNetBlock(128,256),
            nn.MaxPool1d(2),        # out 1 x 256 x n//2
            ResNetBlock(256,256),
            ResNetBlock(256,256),
            ResNetBlock(256,512),
            nn.MaxPool1d(2),        # out 1 x 512 x n//8
            nn.Flatten(),
            nn.Linear(512*(input//8), 256),
            nn.Dropout(0.5),
            nn.Sigmoid(),
            nn.Linear(256, num_classes),
        )
        self.model.to('cuda:0')

    def forward(self, x):

        return self.model(x)
    
    def train_model(self, train_loader, valid_loader, num_epochs = 5, learning_rate=0.001, save_best = False, save_thr = 0.94):
        best_accuracy = 0.0
        total_step = len(train_loader)
        # Loss and optimizer
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.RMSprop(self.parameters(), lr=learning_rate, weight_decay = 0.005, momentum = 0.9)  

        for epoch in range(num_epochs):
            # self.train()
            correct = 0
            total = 0
            for i, (images, labels) in enumerate(tqdm(train_loader)):
                # Move tensors to the configured device
                images = images.float().to("cuda")
                labels = labels.type(torch.LongTensor)
                labels = labels.to("cuda")


                optimizer.zero_grad()

                # Forward pass
                outputs = self.forward(images)
                loss = criterion(outputs, labels)
                # Backward and optimize
                loss.backward()
                
                optimizer.step()

                # accuracy
                _, predicted = torch.max(outputs.data, 1)
                correct += (torch.eq(predicted, labels)).sum().item()
                total += labels.size(0)

                del images, labels, outputs

            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}, Accuracy: {:.4f}'
                            .format(epoch+1, num_epochs, i+1, total_step, loss.item(), (float(correct))/total))


            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Validation
            with torch.no_grad():
                correct = 0
                total = 0
                for images, labels in valid_loader:
                    images = images.float().to("cuda")
                    labels = labels.to("cuda")
                    outputs = self.forward(images)
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (torch.eq(predicted, labels)).sum().item()
                    del images, labels, outputs
                if(((100 * correct / total) > best_accuracy) and save_best and ((100 * correct / total) > save_thr)):
                    torch.save(self.state_dict(), "best_resnet50_MINST-DVS2.pt")

                print('Accuracy of the network: {} %'.format( 100 * correct / total))

In [14]:
model_res = ResNetLike()


In [10]:
model_res.train_model(train,val,num_epochs=40)

100%|██████████| 34522/34522 [10:43<00:00, 53.67it/s]


Epoch [1/40], Step [34522/34522], Loss: 0.5488, Accuracy: 0.6394
Accuracy of the network: 64.55022431754239 %


100%|██████████| 34522/34522 [10:13<00:00, 56.30it/s]


Epoch [2/40], Step [34522/34522], Loss: 0.5467, Accuracy: 0.6494
Accuracy of the network: 59.71191761625949 %


100%|██████████| 34522/34522 [10:10<00:00, 56.53it/s]


...

Epoch [4/40], Step [34522/34522], Loss: 0.6205, Accuracy: 0.6404
Accuracy of the network: 56.61528545719862 %


100%|██████████| 34522/34522 [10:42<00:00, 53.70it/s]


Epoch [5/40], Step [34522/34522], Loss: 0.7793, Accuracy: 0.6412
Accuracy of the network: 66.2835706862777 %


 13%|█▎        | 4586/34522 [01:26<09:21, 53.30it/s]


KeyboardInterrupt: 

In [15]:
# torch.save(model_res.state_dict(), "best_resnet50_afdb_filtration_20Hz.pt")
model_res.load_state_dict(torch.load("best_resnet50_afdb_filtration_20Hz.pt", weights_only=True))

<All keys matched successfully>

In [12]:
model_res.train_model(train,val,num_epochs=90,learning_rate=0.0001,save_best=True)

  0%|          | 0/34522 [00:00<?, ?it/s]

100%|██████████| 34522/34522 [10:05<00:00, 57.01it/s]


Epoch [1/90], Step [34522/34522], Loss: 0.2227, Accuracy: 0.9090
Accuracy of the network: 92.63354950374949 %


100%|██████████| 34522/34522 [08:08<00:00, 70.69it/s]


Epoch [2/90], Step [34522/34522], Loss: 0.0504, Accuracy: 0.9299
Accuracy of the network: 92.11539227507595 %


...

Epoch [7/90], Step [34522/34522], Loss: 0.0697, Accuracy: 0.9347
Accuracy of the network: 93.76400853094641 %


100%|██████████| 34522/34522 [06:12<00:00, 92.79it/s]


Epoch [8/90], Step [34522/34522], Loss: 0.0823, Accuracy: 0.9348
Accuracy of the network: 93.71078063953131 %


  1%|          | 398/34522 [00:04<06:13, 91.30it/s]


KeyboardInterrupt: 

In [16]:
model_res.to("cpu")
torch.save(model_res.state_dict(), "best_resnet50_afdb_filtration_20Hz_cpu.pt")